# Image ↔ text association probe

Pick one image and a list of sentences; get the CLIP cosine similarity of the image against each.

Everything is in the **CONFIG** cell below — edit it and re-run the last cell.

**Reading the numbers.** Raw CLIP cosines are compressed into a narrow band (typically 0.15–0.35) and
their absolute value means very little: what carries information is the *ranking* and the *gap* between
sentences. The softmax column applies CLIP's own learned temperature (`logit_scale`, ≈100) to the same
cosines, which is exactly how CLIP is used for zero-shot classification — that is the number to quote
when you ask "does the model prefer A or B?". It is a distribution over *your* sentence list, so it
changes if you add or remove a sentence.


## Config

In [ ]:

CONFIG = {
    # An image path. Accepts a full path, a path relative to this notebook, a bare filename
    # ("000000000139.jpg") or just the COCO id ("139") — resolved inside IMAGE_DIR.
    "image": "000000000139.jpg",
    "image_dir": "../../val2017",

    "sentences": [
        "A photo of a person.",
        "A photo that does not show a person.",
        "A photo of a television.",
        "A photo that does not show a television.",
        "A living room with a couch.",
        "A living room without a couch.",
    ],

    # --- retrieval (section below) -------------------------------------------------
    "query": "a living room without a couch",
    "top_k": 10,
    "grid_cols": 5,
    "index_batch_size": 64,     # images per forward pass when building the index

    "model_name": "openai/clip-vit-base-patch32",
    "device": None,        # None -> auto (mps / cuda / cpu)
    "figsize": (12.5, 4.6),
    "max_label_chars": 58,  # truncation of long sentences on the chart only
}

## Setup (run once)

In [ ]:

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from transformers import AutoProcessor, CLIPModel

plt.rcParams.update({"figure.dpi": 120, "font.size": 9})


def pick_device(requested=None) -> str:
    if requested:
        return requested
    if torch.cuda.is_available():
        return "cuda"
    if getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
        return "mps"
    return "cpu"


DEVICE = pick_device(CONFIG["device"])
processor = AutoProcessor.from_pretrained(CONFIG["model_name"])
model = CLIPModel.from_pretrained(CONFIG["model_name"]).to(DEVICE).eval()
print(f"{CONFIG['model_name']} on {DEVICE}")


def _as_tensor(out):
    """transformers <5 returns a tensor from get_*_features; >=5 returns an output object."""
    if isinstance(out, torch.Tensor):
        return out
    for attr in ("image_embeds", "text_embeds", "pooler_output"):
        v = getattr(out, attr, None)
        if v is not None:
            return v
    return out[0]


def l2n(x: torch.Tensor) -> torch.Tensor:
    return x / x.norm(dim=-1, keepdim=True).clamp_min(1e-12)


def resolve_image(spec, image_dir) -> Path:
    """Full path, relative path, bare filename, or a COCO id like 139 / '139'."""
    p = Path(str(spec)).expanduser()
    if p.is_file():
        return p
    d = Path(image_dir).expanduser()
    for cand in (d / str(spec), d / f"{spec}.jpg", d / f"{int(spec):012d}.jpg" if str(spec).isdigit() else d / str(spec)):
        if cand.is_file():
            return cand
    raise FileNotFoundError(f"{spec!r} not found directly nor inside {d}")


@torch.no_grad()
def embed_image(path) -> np.ndarray:
    img = Image.open(path).convert("RGB")
    inputs = processor(images=img, return_tensors="pt").to(DEVICE)
    return l2n(_as_tensor(model.get_image_features(**inputs))).float().cpu().numpy()[0]


@torch.no_grad()
def embed_texts(texts) -> np.ndarray:
    inputs = processor(text=list(texts), padding=True, truncation=True, max_length=77,
                       return_tensors="pt").to(DEVICE)
    return l2n(_as_tensor(model.get_text_features(**inputs))).float().cpu().numpy()


LOGIT_SCALE = float(model.logit_scale.detach().exp())
print(f"logit_scale = {LOGIT_SCALE:.1f}")

## Probe

In [ ]:

def probe(image=None, sentences=None, show=True) -> pd.DataFrame:
    """Cosine similarity between one image and each sentence, ranked."""
    image = CONFIG["image"] if image is None else image
    sentences = CONFIG["sentences"] if sentences is None else list(sentences)

    path = resolve_image(image, CONFIG["image_dir"])
    v_img = embed_image(path)
    V_txt = embed_texts(sentences)

    cos = V_txt @ v_img                       # both are unit-norm -> dot product is the cosine
    logits = LOGIT_SCALE * cos
    prob = np.exp(logits - logits.max())
    prob /= prob.sum()

    df = (pd.DataFrame({"sentence": sentences, "cosine": cos, "softmax": prob})
            .sort_values("cosine", ascending=False)
            .assign(rank=lambda d: np.arange(1, len(d) + 1))
            .set_index("rank"))

    if show:
        img = Image.open(path).convert("RGB")
        fig, ax = plt.subplots(1, 2, figsize=CONFIG["figsize"],
                               gridspec_kw={"width_ratios": [1, 1.35]})
        ax[0].imshow(img)
        ax[0].axis("off")
        ax[0].set_title(path.name, fontsize=8)

        d = df.iloc[::-1]                     # best at the top of a horizontal bar chart
        y = np.arange(len(d))
        cut = CONFIG["max_label_chars"]
        labels = [s if len(s) <= cut else s[:cut - 1] + "…" for s in d["sentence"]]
        colors = ["#C44E52" if i == len(d) - 1 else "#4C72B0" for i in range(len(d))]
        ax[1].barh(y, d["cosine"], color=colors, height=0.62)
        ax[1].set_yticks(y, labels, fontsize=7.5)
        for i, (c, p) in enumerate(zip(d["cosine"], d["softmax"])):
            ax[1].text(c + 0.002, i, f"{c:.3f}  ({p:.0%})", va="center", fontsize=7)
        lo, hi = float(d["cosine"].min()), float(d["cosine"].max())
        pad = max((hi - lo) * 0.35, 0.01)
        ax[1].set_xlim(lo - pad * 0.4, hi + pad * 2.2)
        ax[1].set_xlabel("cosine similarity")
        ax[1].set_title("red = best match", fontsize=8)
        ax[1].grid(axis="x", alpha=0.25)
        ax[1].spines[["top", "right"]].set_visible(False)
        fig.tight_layout()
        plt.show()

    return df


probe()

## Optional — several images at once

A matrix is what you want when the question is "does this sentence pick out the right image?" rather
than "which sentence fits this image?". Rows are images, columns are sentences, values are cosines;
the softmax is taken **along each row**, i.e. still a distribution over sentences.


In [ ]:

def probe_grid(images, sentences=None, thumb=1.7) -> pd.DataFrame:
    sentences = CONFIG["sentences"] if sentences is None else list(sentences)
    paths = [resolve_image(i, CONFIG["image_dir"]) for i in images]
    V_img = np.vstack([embed_image(p) for p in paths])
    V_txt = embed_texts(sentences)
    C = V_img @ V_txt.T

    n = len(paths)
    fig, axes = plt.subplots(1, n, figsize=(thumb * n, thumb * 1.15))
    for a, p in zip(np.atleast_1d(axes), paths):
        a.imshow(Image.open(p).convert("RGB"))
        a.axis("off")
        a.set_title(p.stem[-6:], fontsize=7)
    fig.tight_layout()
    plt.show()

    df = pd.DataFrame(C, index=[p.stem[-6:] for p in paths], columns=sentences)
    return df.style.background_gradient(cmap="RdYlBu_r", axis=1).format("{:.3f}")


# probe_grid(["000000000139.jpg", "000000000285.jpg", "000000000632.jpg"])

---
# Retrieval over the whole folder

Encode every image in `image_dir` once, then rank all of them against a text query.

The index is cached on disk under `data/cache/`, keyed by the model **and** the exact list of filenames,
so it is built once and reloaded instantly afterwards. Adding or removing an image from the folder
changes the key and triggers a rebuild — the index can never silently go out of sync with the folder.


In [ ]:
import hashlib
import time

CACHE_DIR = Path("../data/cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def list_images(image_dir=None) -> list[Path]:
    d = Path(image_dir or CONFIG["image_dir"]).expanduser()
    if not d.is_dir():
        raise NotADirectoryError(d)
    return sorted(p for p in d.iterdir() if p.suffix.lower() in IMAGE_EXTS)


@torch.no_grad()
def build_index(image_dir=None, batch_size=None, force=False):
    """(paths, unit-norm embeddings) for every image in the folder, cached on disk."""
    paths = list_images(image_dir)
    batch_size = batch_size or CONFIG["index_batch_size"]
    key = hashlib.sha1(
        (CONFIG["model_name"] + "\x00" + "\x00".join(p.name for p in paths)).encode()
    ).hexdigest()[:12]
    cache = CACHE_DIR / f"imgindex_{key}.npz"

    if cache.exists() and not force:
        z = np.load(cache, allow_pickle=True)
        print(f"index loaded from {cache.name} — {len(z['names'])} images")
        return [Path(image_dir or CONFIG["image_dir"]).expanduser() / n for n in z["names"]], z["e"]

    print(f"building index over {len(paths)} images (one-off)…")
    t0 = time.time()
    out = []
    for start in range(0, len(paths), batch_size):
        chunk = paths[start:start + batch_size]
        imgs = [Image.open(p).convert("RGB") for p in chunk]
        inputs = processor(images=imgs, return_tensors="pt").to(DEVICE)
        out.append(l2n(_as_tensor(model.get_image_features(**inputs))).float().cpu().numpy())
        done = min(start + batch_size, len(paths))
        rate = done / (time.time() - t0)
        print(f"\r  {done}/{len(paths)}  ({rate:.0f} img/s, "
              f"eta {(len(paths) - done) / max(rate, 1e-6):.0f}s)", end="", flush=True)
    E = np.vstack(out).astype(np.float32)
    print(f"\n  done in {time.time() - t0:.1f}s")

    np.savez_compressed(cache, e=E, names=np.array([p.name for p in paths]))
    return paths, E


INDEX_PATHS, INDEX_EMB = build_index()
print(f"index: {INDEX_EMB.shape[0]} images x {INDEX_EMB.shape[1]} dims")

In [ ]:
def search(query=None, k=None, cols=None, show=True) -> pd.DataFrame:
    """Rank every indexed image against a text query and show the top-k."""
    query = CONFIG["query"] if query is None else query
    k = k or CONFIG["top_k"]
    cols = cols or CONFIG["grid_cols"]

    v = embed_texts([query])[0]
    cos = INDEX_EMB @ v                      # unit-norm on both sides
    order = np.argsort(-cos)[:k]

    df = pd.DataFrame({
        "file": [INDEX_PATHS[i].name for i in order],
        "cosine": cos[order],
    }).assign(rank=lambda d: np.arange(1, len(d) + 1)).set_index("rank")

    if show:
        rows = int(np.ceil(len(order) / cols))
        fig, axes = plt.subplots(rows, cols, figsize=(2.35 * cols, 2.65 * rows))
        axes = np.atleast_1d(axes).ravel()
        for a in axes:
            a.axis("off")
        for a, i, r in zip(axes, order, range(1, len(order) + 1)):
            a.imshow(Image.open(INDEX_PATHS[i]).convert("RGB"))
            a.set_title(f"#{r}  {cos[i]:.3f}\n{INDEX_PATHS[i].stem[-6:]}", fontsize=7.5)
        fig.suptitle(f'"{query}"   —   top {len(order)} of {len(INDEX_PATHS)}', fontsize=10)
        fig.tight_layout(rect=[0, 0, 1, 0.96])
        plt.show()

    return df


search()

### Negation stress-test

The pair that matters for this project: the same scene, queried with and without the negation. If the
two result sets overlap heavily, the model is ignoring the negation — that overlap is a measurable
retrieval-side version of the affirmation bias, on real images rather than on a benchmark's multiple
choices.


In [ ]:
def compare_queries(q_pos, q_neg, k=None):
    """Run two queries and report how much their top-k overlap."""
    k = k or CONFIG["top_k"]
    a, b = search(q_pos, k=k), search(q_neg, k=k)
    sa, sb = set(a["file"]), set(b["file"])
    inter = sa & sb
    print(f'overlap of the two top-{k}: {len(inter)}/{k} images ({len(inter) / k:.0%})')
    v = embed_texts([q_pos, q_neg])
    print(f"cosine between the two query embeddings: {float(v[0] @ v[1]):.3f}")
    if inter:
        print("shared:", ", ".join(sorted(inter)))
    return a, b


# compare_queries("a living room with a couch", "a living room without a couch")